# Introdução Teórica ao Apache Spark

O **Apache Spark** é uma plataforma de computação em cluster unificada, rápida e de uso geral para processamento de dados em larga escala. Ele foi projetado para ser altamente veloz, utilizando computação em memória para evitar gargalos de E/S de disco.

### Arquitetura Distribuída do Spark

A arquitetura do Spark segue o modelo **Master-Slave** (Mestre-Escravo), composto pelos seguintes componentes principais:

1. **Driver Program (Programa Driver):**
   - É o processo central que executa a função `main()` da aplicação e cria o `SparkContext` ou `SparkSession`.
   - É responsável por converter o código do usuário em tarefas físicas distribuídas (um Grafo Acíclico Dirigido ou DAG).
   - Agenda e coordena a execução de tarefas nos Workers.

2. **Cluster Manager (Gerenciador de Cluster):**
   - Aloca recursos físicos entre as aplicações (pode ser o gerenciador nativo do Spark, YARN, Mesos ou Kubernetes).

3. **Worker Nodes e Executors (Trabalhadores e Executores):**
   - **Worker Nodes:** São as máquinas físicas ou virtuais do cluster que disponibilizam recursos computacionais.
   - **Executors:** São processos iniciados nos Worker Nodes para uma aplicação Spark específica. Eles são responsáveis por executar as tarefas (`Tasks`) individuais de processamento de dados e manter os dados armazenados em memória ou disco (cache).

---

### RDD vs. DataFrame estruturado

* **RDD (Resilient Distributed Dataset):**
  - É a abstração original e fundamental do Spark.
  - Representa uma coleção imutável e distribuída de objetos que podem ser processados em paralelo.
  - Oferece controle de baixo nível, mas carece de otimizações automáticas de engine, pois o Spark não conhece a estrutura interna dos dados contidos no RDD.

* **DataFrame:**
  - É uma coleção distribuída de dados organizados em colunas nomeadas, conceitualmente equivalente a uma tabela de banco de dados relacional ou a um DataFrame do pandas, mas com otimizações robustas sob o capô.
  - Utiliza o otimizador **Catalyst** e a engine de execução **Tungsten**, que traduzem e otimizam as consultas em planos físicos altamente eficientes, proporcionando maior desempenho e menor consumo de memória em relação aos RDDs tradicionais.

In [7]:
!pip install pyspark -q

In [8]:
from pyspark.sql import SparkSession

# Inicializa a sessão Spark no Google Colab
spark = SparkSession.builder \
    .appName('Guia_Spark_Colab') \
    .getOrCreate()

# Exibe o objeto da sessão criado para validar o sucesso
print("Sessão Spark iniciada com sucesso!")
spark

Sessão Spark iniciada com sucesso!


# Operações Básicas com DataFrames

No Apache Spark, um **DataFrame** é uma coleção distribuída de dados organizados em colunas nomeadas. Ao contrário dos RDDs, os DataFrames possuem um esquema conhecido, o que permite ao Spark aplicar otimizações altamente eficientes por meio do otimizador Catalyst.

### Estruturando Dados no PySpark
Para iniciar o trabalho com DataFrames, podemos criá-los a partir de diversas fontes, como arquivos CSV, JSON, bancos de dados relacionais ou mesmo convertendo listas e dicionários Python locais.

Uma vez criado o DataFrame, dispomos de métodos fundamentais para explorar sua estrutura e conteúdo:
* `show()`: Exibe de forma tabular as primeiras linhas do DataFrame.
* `printSchema()`: Mostra graficamente o esquema das colunas (nomes e tipos de dados, além de informações sobre nulidade).
* `describe()`: Computa estatísticas descritivas para colunas numéricas e de string (como contagem, média, desvio padrão, valores mínimo e máximo).

In [9]:
# 1. Definição dos dados fictícios (Tuplas) e nomes das colunas
dados_funcionarios = [
    (1, "Ana Souza", 28, "Tecnologia"),
    (2, "Bruno Lima", 35, "Recursos Humanos"),
    (3, "Carlos Mendes", 42, "Financeiro"),
    (4, "Daniela Reis", 24, "Tecnologia"),
    (5, "Eduardo Silva", 31, "Marketing")
]

colunas = ["id", "nome", "idade", "departamento"]

# 2. Criação do DataFrame utilizando a sessão ativa do Spark
df_funcionarios = spark.createDataFrame(dados_funcionarios, schema=colunas)

# 3. Exibição do conteúdo do DataFrame
print("=== Conteúdo do DataFrame ===")
df_funcionarios.show()

# 4. Exibição do esquema do DataFrame
print("=== Esquema do DataFrame (printSchema) ===")
df_funcionarios.printSchema()

# 5. Exibição das estatísticas descritivas básicas das colunas numéricas
print("=== Estatísticas Descritivas (describe) ===")
df_funcionarios.describe("idade").show()

=== Conteúdo do DataFrame ===
+---+-------------+-----+----------------+
| id|         nome|idade|    departamento|
+---+-------------+-----+----------------+
|  1|    Ana Souza|   28|      Tecnologia|
|  2|   Bruno Lima|   35|Recursos Humanos|
|  3|Carlos Mendes|   42|      Financeiro|
|  4| Daniela Reis|   24|      Tecnologia|
|  5|Eduardo Silva|   31|       Marketing|
+---+-------------+-----+----------------+

=== Esquema do DataFrame (printSchema) ===
root
 |-- id: long (nullable = true)
 |-- nome: string (nullable = true)
 |-- idade: long (nullable = true)
 |-- departamento: string (nullable = true)

=== Estatísticas Descritivas (describe) ===
+-------+-----------------+
|summary|            idade|
+-------+-----------------+
|  count|                5|
|   mean|             32.0|
| stddev|6.892024376045111|
|    min|               24|
|    max|               42|
+-------+-----------------+



# Transformações, Ações e Lazy Evaluation

No Apache Spark, as operações sobre DataFrames são divididas em duas categorias principais:

1. **Transformações (Lazy Evaluation):**
   * Operações que não computam o resultado imediatamente. Em vez disso, elas criam um novo DataFrame e registram uma lista de instruções no plano de execução (um Grafo Acíclico Dirigido ou DAG).
   * O Spark adia a execução física até que seja estritamente necessário, permitindo que o otimizador Catalyst reorganize e otimize todo o fluxo de trabalho.
   * *Exemplos comuns:* `select()`, `filter()`, `where()`, `groupBy()`, `join()`, `orderBy()`.

2. **Ações:**
   * Operações que disparam a execução real de todas as transformações acumuladas no plano e retornam o resultado final ao programa driver ou gravam os dados em disco.
   * *Exemplos comuns:* `show()`, `count()`, `collect()`, `first()`, `take()`.

---

In [10]:
# 1. Demonstração de Lazy Evaluation (Transformações)
# Vamos selecionar colunas e filtrar os dados. Note que isso não realiza a computação pesada ainda.
print("=== Aplicando Transformações (Lazy Evaluation) ===")
df_filtrado = df_funcionarios.select("nome", "idade", "departamento").filter(df_funcionarios.idade >= 30)

# Exibindo o plano de execução e o tipo do objeto
print(f"Tipo do objeto resultante: {type(df_filtrado)}")
print("Note que nenhuma computação foi feita nos dados até agora; apenas o plano de execução foi criado.\n")

# 2. Executando as Transformações com uma Ação (show e count)
print("=== Executando as Ações (Disparando a Computação) ===")
# Ação show() força o Spark a computar e exibir o resultado das transformações
df_filtrado.show()

# Ação count() retorna um número inteiro de volta para o driver python
total_filtrado = df_filtrado.count()
print(f"Total de funcionários com 30 anos ou mais: {total_filtrado}\n")

# 3. Agrupamento (groupBy) e Ação associada
print("=== Agrupando dados por Departamento (Transformação + Ação) ===")
# groupBy é uma transformação, e count() ou show() executa a agregação
df_agrupado = df_funcionarios.groupBy("departamento").count()

print("Resultado do agrupamento:")
df_agrupado.show()

# 4. Uso do método collect() para trazer os resultados em uma lista local do Python
print("=== Usando collect() para coletar os dados no Driver ===")
lista_resultados = df_filtrado.collect()
for linha in lista_resultados:
    print(f"Funcionário: {linha['nome']}, Idade: {linha['idade']}, Departamento: {linha['departamento']}")

=== Aplicando Transformações (Lazy Evaluation) ===
Tipo do objeto resultante: <class 'pyspark.sql.classic.dataframe.DataFrame'>
Note que nenhuma computação foi feita nos dados até agora; apenas o plano de execução foi criado.

=== Executando as Ações (Disparando a Computação) ===
+-------------+-----+----------------+
|         nome|idade|    departamento|
+-------------+-----+----------------+
|   Bruno Lima|   35|Recursos Humanos|
|Carlos Mendes|   42|      Financeiro|
|Eduardo Silva|   31|       Marketing|
+-------------+-----+----------------+

Total de funcionários com 30 anos ou mais: 3

=== Agrupando dados por Departamento (Transformação + Ação) ===
Resultado do agrupamento:
+----------------+-----+
|    departamento|count|
+----------------+-----+
|Recursos Humanos|    1|
|      Tecnologia|    2|
|      Financeiro|    1|
|       Marketing|    1|
+----------------+-----+

=== Usando collect() para coletar os dados no Driver ===
Funcionário: Bruno Lima, Idade: 35, Departamento: R

# Spark SQL e Consultas Declarativas

O módulo **Spark SQL** permite que você utilize a força das consultas declarativas SQL diretamente sobre os DataFrames do Spark. Isso possibilita que analistas de dados e engenheiros que já dominam a linguagem SQL possam interagir com grandes volumes de dados de forma familiar e altamente otimizada.

### Views Temporárias
Para executar consultas SQL estruturadas, primeiro precisamos registrar o DataFrame como uma **view temporária** na sessão ativa do Spark.

* **`createOrReplaceTempView(name)`**: Cria (ou substitui, caso já exista) uma view temporária local com o nome especificado. Essa view fica disponível apenas durante o tempo de vida da sessão Spark atual e não persiste no armazenamento físico.

In [11]:
# 1. Registrar o DataFrame existente como uma view temporária
df_funcionarios.createOrReplaceTempView("view_funcionarios")
print("View temporária 'view_funcionarios' registrada com sucesso!\n")

# 2. Escrever e executar uma consulta SQL pura
# Selecionar nome e departamento dos funcionários do departamento de 'Tecnologia'
query_sql = """
    SELECT nome, departamento
    FROM view_funcionarios
    WHERE departamento = 'Tecnologia'
"""

# spark.sql() retorna um novo DataFrame com o resultado da consulta
df_resultado_sql = spark.sql(query_sql)

# 3. Exibir o resultado utilizando o método .show()
print("=== Resultado da Consulta SQL (Departamento de Tecnologia) ===")
df_resultado_sql.show()

View temporária 'view_funcionarios' registrada com sucesso!

=== Resultado da Consulta SQL (Departamento de Tecnologia) ===
+------------+------------+
|        nome|departamento|
+------------+------------+
|   Ana Souza|  Tecnologia|
|Daniela Reis|  Tecnologia|
+------------+------------+



# Exercício Prático Consolidado

Nesta seção, vamos consolidar todos os conceitos apresentados (criação de DataFrames, exploração de esquemas, transformações, agregações e ordenações) em um mini estudo de caso prático.

### Cenário de Negócio: Análise de Vendas de E-commerce

Imagine que você recebeu um conjunto de dados brutos com as vendas recentes de uma loja online. Cada venda possui informações sobre o identificador do pedido, nome do produto, categoria, valor unitário e quantidade vendida.

Sua missão é realizar uma análise de desempenho por categoria de produto. Para isso, você deverá:
1. Criar um novo DataFrame com os dados de vendas fictícios.
2. Adicionar uma nova coluna calculando o valor total de cada venda (`valor * quantidade`).
3. Filtrar as vendas para remover possíveis transações com quantidades menores ou iguais a zero.
4. Agrupar as vendas por categoria e calcular o faturamento total acumulado em cada uma delas.
5. Ordenar o resultado final de forma decrescente pelo faturamento total, destacando a categoria campeã de vendas.

In [12]:
from pyspark.sql.functions import col, sum as _sum

# 1. Definição dos dados fictícios de vendas (id_venda, produto, categoria, valor, quantidade)
dados_vendas = [
    (101, "Notebook Pro", "Eletrônicos", 4500.0, 2),
    (102, "Smartphone X", "Eletrônicos", 2500.0, 3),
    (103, "Cadeira Ergonômica", "Móveis", 1200.0, 5),
    (104, "Mesa Digitalizadora", "Eletrônicos", 600.0, 0),  # Venda com quantidade zero (será filtrada)
    (105, "Livro de Spark", "Livros", 80.0, 10),
    (106, "Teclado Mecânico", "Eletrônicos", 350.0, 4),
    (107, "Livro de Python", "Livros", 60.0, 15),
    (108, "Sofá Retrátil", "Móveis", 2200.0, 1)
]

colunas_vendas = ["id_venda", "produto", "categoria", "valor", "quantidade"]

# Criando o DataFrame
df_vendas = spark.createDataFrame(dados_vendas, schema=colunas_vendas)

print("=== DataFrame Original de Vendas ===")
df_vendas.show()

# 2 e 3. Processamento e transformações encadeadas
# - Adiciona coluna de valor_total (valor * quantidade)
# - Filtra transações onde a quantidade é maior do que zero
# - Agrupa por categoria de produto
# - Calcula a soma do faturamento total
# - Ordena o faturamento total em ordem decrescente

print("=== Processando Análise de Faturamento por Categoria ===")
df_analise_categoria = df_vendas \
    .withColumn("valor_total", col("valor") * col("quantidade")) \
    .filter(col("quantidade") > 0) \
    .groupBy("categoria") \
    .agg(_sum("valor_total").alias("faturamento_total")) \
    .orderBy(col("faturamento_total").desc())

# 4. Exibição do resultado final consolidado
df_analise_categoria.show()


=== DataFrame Original de Vendas ===
+--------+-------------------+-----------+------+----------+
|id_venda|            produto|  categoria| valor|quantidade|
+--------+-------------------+-----------+------+----------+
|     101|       Notebook Pro|Eletrônicos|4500.0|         2|
|     102|       Smartphone X|Eletrônicos|2500.0|         3|
|     103| Cadeira Ergonômica|     Móveis|1200.0|         5|
|     104|Mesa Digitalizadora|Eletrônicos| 600.0|         0|
|     105|     Livro de Spark|     Livros|  80.0|        10|
|     106|   Teclado Mecânico|Eletrônicos| 350.0|         4|
|     107|    Livro de Python|     Livros|  60.0|        15|
|     108|      Sofá Retrátil|     Móveis|2200.0|         1|
+--------+-------------------+-----------+------+----------+

=== Processando Análise de Faturamento por Categoria ===
+-----------+-----------------+
|  categoria|faturamento_total|
+-----------+-----------------+
|Eletrônicos|          17900.0|
|     Móveis|           8200.0|
|     Livros|

# Resumo Executivo da Primeira Aula

### Q&A

* **Como funciona a arquitetura do Apache Spark?**
  O Spark opera em um modelo master-nodes composto pelo *Driver Program* (responsável por coordenar o fluxo e converter o código em tarefas físicas distribuídas) e os *Worker Nodes* (onde residem os *Executors*, processos que executam as tarefas de processamento de dados e mantêm o armazenamento em cache).

* **Qual é a diferença fundamental entre RDDs e DataFrames?**
  Enquanto os RDDs fornecem controle de baixo nível sobre coleções imutáveis sem otimizações automáticas de dados estruturados, os DataFrames estruturam dados em colunas nomeadas e se beneficiam do otimizador **Catalyst** e da engine **Tungsten**, que geram planos físicos de execução altamente eficientes e velozes.

* **O que é o conceito de *Lazy Evaluation*?**
  É o adiamento da computação física dos dados. Operações de **Transformação** (como `filter` ou `select`) apenas criam e registram um plano de execução (DAG) sem processar nada de imediato. Os dados só são efetivamente processados quando uma operação de **Ação** (como `show`, `count` ou `collect`) é invocada.

---

### Resultado da Análise de Dados

* **Análise de Faturamento por Categoria (Estudo de Caso):**
  * A categoria de **Eletrônicos** liderou com o maior faturamento total acumulado, somando \$17.900,00 de receita válida.
  * A categoria de **Móveis** ficou na segunda posição com um faturamento total de \$8.200,00.
  * A categoria de **Livros** registrou o menor faturamento consolidado entre as vendas válidas, totalizando \$1.700,00.
  * O pipeline removeu com sucesso transações inválidas do cálculo, filtrando registros que continham quantidades zeradas (como o produto *Mesa Digitalizadora*, que possuía quantidade 0).

---

### Insights or Next Steps

O principal insight é que Apache Spark permite processar grandes volumes de dados distribuindo o trabalho, mas usá-lo bem exige entender como suas operações são executadas.
Para destacar na sua aula, eu priorizaria estes pontos:
1. Spark é um mecanismo de processamento; PySpark é sua API Python. Você escreve em Python, enquanto o Spark coordena a execução dos dados.
2. É possível começar localmente. Os conceitos e a API podem ser ensinados em uma única máquina. Um cluster permite distribuir o processamento entre várias máquinas.
3. DataFrames são o melhor ponto de entrada. Eles organizam dados em linhas, colunas e tipos, com operações próximas de SQL e pandas.
4. Descrever uma operação e executá-la são etapas diferentes. Transformações como filter e select constroem um plano; ações como count, show e escrita de arquivos solicitam resultados. Essa avaliação preguiçosa permite otimizar o plano.
5. Mover dados pode custar mais do que calculá-los. Agregações, ordenações e joins frequentemente exigem shuffle, redistribuindo dados entre partições. Esse movimento é um dos principais fatores de desempenho.
6. O driver pode virar um gargalo. collect() e toPandas() trazem dados para sua memória. Um processamento distribuído pode falhar se o resultado inteiro não couber nela.
7. Qualidade dos dados continua sendo essencial. Spark não resolve automaticamente nulos, duplicatas ou chaves inconsistentes. Um join com chaves repetidas pode multiplicar linhas e inflar indicadores.
8. Spark não é sempre a ferramenta mais rápida ou simples. Para dados pequenos, Python ou pandas podem ser suficientes. O benefício cresce quando volume, complexidade ou necessidade de distribuição justificam seu custo.
